In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/risk_scorecard.csv")

In [4]:
df.head()

,loan_id,fico_avg,dti,revol_util,fico_points,dti_points,revol_points,risk_score,risk_tier,default_flag
0,68407277,677.0,5.91,29.7,30,0,0,30,Medium,0
1,68355089,717.0,16.06,19.2,15,0,0,15,Low,0
2,68341763,697.0,10.78,56.2,30,0,10,40,Medium,0
3,68476807,697.0,25.37,64.5,30,15,20,65,Very High,0
4,68426831,692.0,10.20,68.4,30,0,20,50,High,0


In [5]:
df.shape

(1345350, 10)

In [6]:
df.columns.tolist()

['loan_id',
 'fico_avg',
 'dti',
 'revol_util',
 'fico_points',
 'dti_points',
 'revol_points',
 'risk_score',
 'risk_tier',
 'default_flag']

## Calculate Observed Probability of Default (PD)

PD is calculated as observed defaults divided by total loans within each risk tier.

In [7]:
pd_by_tier = (
    df.groupby("risk_tier")
      .agg(
          loans=("default_flag", "size"),
          defaults=("default_flag", "sum")
      )
      .reset_index()
)

pd_by_tier

,risk_tier,loans,defaults
0,High,436292,99026
1,Low,172829,19197
2,Medium,541145,94452
3,Very High,195084,55924


In [8]:
pd_by_tier["PD"] = (
    pd_by_tier["defaults"] / pd_by_tier["loans"]
)

pd_by_tier

,risk_tier,loans,defaults,PD
0,High,436292,99026,0.226972
1,Low,172829,19197,0.111075
2,Medium,541145,94452,0.174541
3,Very High,195084,55924,0.286666


In [9]:
pd_by_tier["PD_pct"] = (pd_by_tier["PD"] * 100).round(2)

pd_by_tier

,risk_tier,loans,defaults,PD,PD_pct
0,High,436292,99026,0.226972,22.70
1,Low,172829,19197,0.111075,11.11
2,Medium,541145,94452,0.174541,17.45
3,Very High,195084,55924,0.286666,28.67


## Assign PD to Each Loan

Each loan receives the observed PD associated with its risk tier.

In [10]:
pd_map = pd_by_tier.set_index("risk_tier")["PD"]

pd_map

risk_tier
High         0.226972
Low          0.111075
Medium       0.174541
Very High    0.286666
Name: PD, dtype: float64

In [11]:
df["PD"] = df["risk_tier"].map(pd_map)

df[["loan_id", "risk_score", "risk_tier", "default_flag", "PD"]].head(10)

,loan_id,risk_score,risk_tier,default_flag,PD
0,68407277,30,Medium,0,0.174541
1,68355089,15,Low,0,0.111075
2,68341763,40,Medium,0,0.174541
3,68476807,65,Very High,0,0.286666
4,68426831,50,High,0,0.226972
5,68476668,50,High,0,0.226972
6,67275481,15,Low,0,0.111075
7,68466926,40,Medium,0,0.174541
8,68616873,40,Medium,0,0.174541
9,68338832,50,High,0,0.226972


## Loss Given Default (LGD)

LGD measures the proportion of exposure expected to be lost if a borrower defaults.

LGD = 1 - Recovery Rate

In [12]:
loans_columns = pd.read_csv("../data/loans_clean.csv", nrows=0)

loans_columns.columns.tolist()

['loan_id',
 'loan_amnt',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'purpose',
 'addr_state',
 'dti',
 'fico_range_low',
 'fico_range_high',
 'open_acc',
 'revol_bal',
 'revol_util',
 'total_acc',
 'default_flag']

In [13]:
check_cols = [
    "recoveries",
    "collection_recovery_fee",
    "total_rec_prncp",
    "out_prncp",
    "funded_amnt",
    "loan_amnt"
]

[col for col in check_cols if col in loans_columns.columns]

['loan_amnt']

### LGD Assumption

The cleaned dataset does not contain recovery information. Therefore, LGD is assumed to be 45%.

This is a modeling assumption, not an observed dataset value.

In [14]:
ASSUMED_LGD = 0.45

df["LGD"] = ASSUMED_LGD

df[["loan_id", "risk_tier", "PD", "LGD"]].head(10)

,loan_id,risk_tier,PD,LGD
0,68407277,Medium,0.174541,0.45
1,68355089,Low,0.111075,0.45
2,68341763,Medium,0.174541,0.45
3,68476807,Very High,0.286666,0.45
4,68426831,High,0.226972,0.45
5,68476668,High,0.226972,0.45
6,67275481,Low,0.111075,0.45
7,68466926,Medium,0.174541,0.45
8,68616873,Medium,0.174541,0.45
9,68338832,High,0.226972,0.45


## Exposure at Default (EAD)

The cleaned dataset does not contain outstanding principal balances. Therefore, the original loan amount is used as the exposure measure for this project.

EAD = Loan Amount

In [15]:
loan_amounts = pd.read_csv(
    "../data/loans_clean.csv",
    usecols=["loan_id", "loan_amnt"]
)

loan_amounts.head()

,loan_id,loan_amnt
0,68407277,3600.0
1,68355089,24700.0
2,68341763,20000.0
3,68476807,10400.0
4,68426831,11950.0


In [16]:
df = df.merge(
    loan_amounts,
    on="loan_id",
    how="left",
    validate="one_to_one"
)

df[["loan_id", "risk_tier", "PD", "LGD", "loan_amnt"]].head(10)

,loan_id,risk_tier,PD,LGD,loan_amnt
0,68407277,Medium,0.174541,0.45,3600.0
1,68355089,Low,0.111075,0.45,24700.0
2,68341763,Medium,0.174541,0.45,20000.0
3,68476807,Very High,0.286666,0.45,10400.0
4,68426831,High,0.226972,0.45,11950.0
5,68476668,High,0.226972,0.45,20000.0
6,67275481,Low,0.111075,0.45,20000.0
7,68466926,Medium,0.174541,0.45,10000.0
8,68616873,Medium,0.174541,0.45,8000.0
9,68338832,High,0.226972,0.45,1400.0


In [17]:
print("Rows:", len(df))
print("Missing loan amounts:", df["loan_amnt"].isna().sum())

Rows: 1345350
Missing loan amounts: 0


### EAD Assumption

Outstanding principal is not available in the cleaned dataset. Therefore, the original loan amount (`loan_amnt`) is used as a proxy for Exposure at Default (EAD).

This is an exposure proxy, not an observed outstanding balance.

In [18]:
df["EAD"] = df["loan_amnt"]

df[["loan_id", "risk_tier", "PD", "LGD", "loan_amnt", "EAD"]].head(10)

,loan_id,risk_tier,PD,LGD,loan_amnt,EAD
0,68407277,Medium,0.174541,0.45,3600.0,3600.0
1,68355089,Low,0.111075,0.45,24700.0,24700.0
2,68341763,Medium,0.174541,0.45,20000.0,20000.0
3,68476807,Very High,0.286666,0.45,10400.0,10400.0
4,68426831,High,0.226972,0.45,11950.0,11950.0
5,68476668,High,0.226972,0.45,20000.0,20000.0
6,67275481,Low,0.111075,0.45,20000.0,20000.0
7,68466926,Medium,0.174541,0.45,10000.0,10000.0
8,68616873,Medium,0.174541,0.45,8000.0,8000.0
9,68338832,High,0.226972,0.45,1400.0,1400.0


In [19]:
print("Missing EAD:", df["EAD"].isna().sum())
print("Minimum EAD:", df["EAD"].min())
print("Maximum EAD:", df["EAD"].max())

Missing EAD: 0
Minimum EAD: 500.0
Maximum EAD: 40000.0


## Expected Credit Loss (ECL)

Expected Credit Loss combines the probability of default, loss severity, and exposure.

ECL = PD × LGD × EAD

In [20]:
df["ECL"] = df["PD"] * df["LGD"] * df["EAD"]

df[
    ["loan_id", "risk_tier", "PD", "LGD", "EAD", "ECL"]
].head(10)

,loan_id,risk_tier,PD,LGD,EAD,ECL
0,68407277,Medium,0.174541,0.45,3600.0,282.756452
1,68355089,Low,0.111075,0.45,24700.0,1234.599836
2,68341763,Medium,0.174541,0.45,20000.0,1570.869176
3,68476807,Very High,0.286666,0.45,10400.0,1341.598081
4,68426831,High,0.226972,0.45,11950.0,1220.541094
5,68476668,High,0.226972,0.45,20000.0,2042.746601
6,67275481,Low,0.111075,0.45,20000.0,999.675980
7,68466926,Medium,0.174541,0.45,10000.0,785.434588
8,68616873,Medium,0.174541,0.45,8000.0,628.347670
9,68338832,High,0.226972,0.45,1400.0,142.992262


In [21]:
print("Missing ECL:", df["ECL"].isna().sum())
print("Minimum ECL:", df["ECL"].min())
print("Maximum ECL:", df["ECL"].max())

Missing ECL: 0
Minimum ECL: 24.991899507605787
Maximum ECL: 5159.992618564311


## Portfolio ECL Analysis

Aggregate expected credit loss to evaluate the overall portfolio and identify where credit risk is concentrated.

In [22]:
total_ecl = df["ECL"].sum()

print(f"Total Portfolio ECL: ${total_ecl:,.2f}")

Total Portfolio ECL: $1,748,555,918.43


In [23]:
total_ead = df["EAD"].sum()

print(f"Total Portfolio EAD: ${total_ead:,.2f}")
print(f"Total Portfolio ECL: ${total_ecl:,.2f}")
print(f"ECL as % of EAD: {(total_ecl / total_ead) * 100:.2f}%")

Total Portfolio EAD: $19,399,906,575.00
Total Portfolio ECL: $1,748,555,918.43
ECL as % of EAD: 9.01%


## ECL by Risk Tier

Aggregate exposure and expected credit loss by risk tier to identify where portfolio credit risk is concentrated.

In [24]:
ecl_by_risk_tier = (
    df.groupby("risk_tier")
      .agg(
          loans=("loan_id", "count"),
          total_ead=("EAD", "sum"),
          total_ecl=("ECL", "sum")
      )
      .reset_index()
)

ecl_by_risk_tier

,risk_tier,loans,total_ead,total_ecl
0,High,436292,6.297072e+09,6.431662e+08
1,Low,172829,2.457208e+09,1.228206e+08
2,Medium,541145,7.743616e+09,6.082104e+08
3,Very High,195084,2.902010e+09,3.743588e+08


In [25]:
ecl_by_risk_tier["ecl_share_pct"] = (
    ecl_by_risk_tier["total_ecl"] / total_ecl * 100
).round(2)

ecl_by_risk_tier

,risk_tier,loans,total_ead,total_ecl,ecl_share_pct
0,High,436292,6.297072e+09,6.431662e+08,36.78
1,Low,172829,2.457208e+09,1.228206e+08,7.02
2,Medium,541145,7.743616e+09,6.082104e+08,34.78
3,Very High,195084,2.902010e+09,3.743588e+08,21.41


In [26]:
ecl_by_risk_tier_display = ecl_by_risk_tier.copy()

ecl_by_risk_tier_display["total_ead"] = (
    ecl_by_risk_tier_display["total_ead"].round(2)
)

ecl_by_risk_tier_display["total_ecl"] = (
    ecl_by_risk_tier_display["total_ecl"].round(2)
)

ecl_by_risk_tier_display

,risk_tier,loans,total_ead,total_ecl,ecl_share_pct
0,High,436292,6.297072e+09,6.431662e+08,36.78
1,Low,172829,2.457208e+09,1.228206e+08,7.02
2,Medium,541145,7.743616e+09,6.082104e+08,34.78
3,Very High,195084,2.902010e+09,3.743588e+08,21.41


## ECL by Loan Purpose

Analyze expected credit loss across different loan purposes to identify portfolio risk concentrations.

In [27]:
loan_purpose = pd.read_csv(
    "../data/loans_clean.csv",
    usecols=["loan_id", "purpose"]
)

loan_purpose.head()

,loan_id,purpose
0,68407277,debt_consolidation
1,68355089,small_business
2,68341763,home_improvement
3,68476807,major_purchase
4,68426831,debt_consolidation


In [28]:
df = df.merge(
    loan_purpose,
    on="loan_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(df))
print("Missing purpose:", df["purpose"].isna().sum())

df[["loan_id", "risk_tier", "ECL", "purpose"]].head(10)

Rows: 1345350
Missing purpose: 0


,loan_id,risk_tier,ECL,purpose
0,68407277,Medium,282.756452,debt_consolidation
1,68355089,Low,1234.599836,small_business
2,68341763,Medium,1570.869176,home_improvement
3,68476807,Very High,1341.598081,major_purchase
4,68426831,High,1220.541094,debt_consolidation
5,68476668,High,2042.746601,debt_consolidation
6,67275481,Low,999.675980,major_purchase
7,68466926,Medium,785.434588,credit_card
8,68616873,Medium,628.347670,credit_card
9,68338832,High,142.992262,other


In [29]:
ecl_by_purpose = (
    df.groupby("purpose")
      .agg(
          loans=("loan_id", "count"),
          total_ead=("EAD", "sum"),
          total_ecl=("ECL", "sum")
      )
      .reset_index()
      .sort_values("total_ecl", ascending=False)
)

ecl_by_purpose

,purpose,loans,total_ead,total_ecl
2,debt_consolidation,780342,1.188647e+10,1.089955e+09
1,credit_card,295285,4.373462e+09,4.046766e+08
4,home_improvement,87507,1.237774e+09,9.908717e+07
9,other,77877,7.654911e+08,6.441184e+07
6,major_purchase,29427,3.483717e+08,2.665598e+07
11,small_business,15416,2.411311e+08,1.930161e+07
7,medical,15556,1.400346e+08,1.179253e+07
0,car,14588,1.290443e+08,1.020643e+07
5,house,7254,1.117219e+08,8.544859e+06
8,moving,9480,7.454552e+07,6.339650e+06


## ECL by Income Segment

Analyze expected credit loss across borrower income groups to identify whether portfolio credit risk is concentrated in specific income segments.

In [30]:
loan_income = pd.read_csv(
    "../data/loans_clean.csv",
    usecols=["loan_id", "annual_inc"]
)

loan_income.head()

,loan_id,annual_inc
0,68407277,55000.0
1,68355089,65000.0
2,68341763,63000.0
3,68476807,104433.0
4,68426831,34000.0


In [31]:
df = df.merge(
    loan_income,
    on="loan_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(df))
print("Missing annual income:", df["annual_inc"].isna().sum())

df[["loan_id", "annual_inc", "risk_tier", "ECL"]].head(10)

Rows: 1345350
Missing annual income: 0


,loan_id,annual_inc,risk_tier,ECL
0,68407277,55000.0,Medium,282.756452
1,68355089,65000.0,Low,1234.599836
2,68341763,63000.0,Medium,1570.869176
3,68476807,104433.0,Very High,1341.598081
4,68426831,34000.0,High,1220.541094
5,68476668,180000.0,High,2042.746601
6,67275481,85000.0,Low,999.675980
7,68466926,85000.0,Medium,785.434588
8,68616873,42000.0,Medium,628.347670
9,68338832,64000.0,High,142.992262


In [32]:
income_bins = [0, 40000, 60000, 80000, 100000, float("inf")]

income_labels = [
    "< $40K",
    "$40K-$60K",
    "$60K-$80K",
    "$80K-$100K",
    "$100K+"
]

df["income_segment"] = pd.cut(
    df["annual_inc"],
    bins=income_bins,
    labels=income_labels,
    right=False
)

df[["annual_inc", "income_segment"]].head(10)

,annual_inc,income_segment
0,55000.0,$40K-$60K
1,65000.0,$60K-$80K
2,63000.0,$60K-$80K
3,104433.0,$100K+
4,34000.0,< $40K
5,180000.0,$100K+
6,85000.0,$80K-$100K
7,85000.0,$80K-$100K
8,42000.0,$40K-$60K
9,64000.0,$60K-$80K


In [33]:
df["income_segment"].value_counts().sort_index()

income_segment
< $40K        210384
$40K-$60K     355541
$60K-$80K     307935
$80K-$100K    194548
$100K+        276942
Name: count, dtype: int64

In [34]:
ecl_by_income = (
    df.groupby("income_segment", observed=True)
    .agg(
        loans=("loan_id", "count"),
        total_ead=("EAD", "sum"),
        total_ecl=("ECL", "sum")
    )
    .reset_index()
)

ecl_by_income

,income_segment,loans,total_ead,total_ecl
0,< $40K,210384,1.692777e+09,1.584786e+08
1,$40K-$60K,355541,4.116091e+09,3.824277e+08
2,$60K-$80K,307935,4.504336e+09,4.110706e+08
3,$80K-$100K,194548,3.347463e+09,3.002038e+08
4,$100K+,276942,5.739239e+09,4.963752e+08


In [35]:
ecl_by_income["ecl_share_pct"] = (
    ecl_by_income["total_ecl"] / ecl_by_income["total_ecl"].sum() * 100
).round(2)

ecl_by_income

,income_segment,loans,total_ead,total_ecl,ecl_share_pct
0,< $40K,210384,1.692777e+09,1.584786e+08,9.06
1,$40K-$60K,355541,4.116091e+09,3.824277e+08,21.87
2,$60K-$80K,307935,4.504336e+09,4.110706e+08,23.51
3,$80K-$100K,194548,3.347463e+09,3.002038e+08,17.17
4,$100K+,276942,5.739239e+09,4.963752e+08,28.39


In [36]:
ecl_by_income["ecl_rate_pct"] = (
    ecl_by_income["total_ecl"] / ecl_by_income["total_ead"] * 100
).round(2)

ecl_by_income

,income_segment,loans,total_ead,total_ecl,ecl_share_pct,ecl_rate_pct
0,< $40K,210384,1.692777e+09,1.584786e+08,9.06,9.36
1,$40K-$60K,355541,4.116091e+09,3.824277e+08,21.87,9.29
2,$60K-$80K,307935,4.504336e+09,4.110706e+08,23.51,9.13
3,$80K-$100K,194548,3.347463e+09,3.002038e+08,17.17,8.97
4,$100K+,276942,5.739239e+09,4.963752e+08,28.39,8.65


### Income Segment Insight

Expected loss rates generally decrease as borrower income increases. Borrowers earning under $40K have the highest ECL rate at 9.36%, while borrowers earning $100K+ have the lowest at 8.65%. However, the $100K+ segment contributes the largest share of total portfolio ECL (28.39%) because of its larger total exposure.

In [37]:
print("Total rows:", len(df))
print("Missing PD:", df["PD"].isna().sum())
print("Missing LGD:", df["LGD"].isna().sum())
print("Missing EAD:", df["EAD"].isna().sum())
print("Missing ECL:", df["ECL"].isna().sum())

print("\nECL formula check:")
print(
    np.allclose(
        df["ECL"],
        df["PD"] * df["LGD"] * df["EAD"]
    )
)

Total rows: 1345350
Missing PD: 0
Missing LGD: 0
Missing EAD: 0
Missing ECL: 0

ECL formula check:
True


In [38]:
output_columns = [
    "loan_id",
    "risk_score",
    "risk_tier",
    "default_flag",
    "PD",
    "LGD",
    "EAD",
    "ECL",
    "purpose",
    "annual_inc",
    "income_segment"
]

final_risk_df = df[output_columns].copy()

final_risk_df.to_csv(
    "../data/credit_risk_ecl.csv",
    index=False
)

print("Export complete!")
print("Rows exported:", len(final_risk_df))

final_risk_df.head()

Export complete!
Rows exported: 1345350


,loan_id,risk_score,risk_tier,default_flag,PD,LGD,EAD,ECL,purpose,annual_inc,income_segment
0,68407277,30,Medium,0,0.174541,0.45,3600.0,282.756452,debt_consolidation,55000.0,$40K-$60K
1,68355089,15,Low,0,0.111075,0.45,24700.0,1234.599836,small_business,65000.0,$60K-$80K
2,68341763,40,Medium,0,0.174541,0.45,20000.0,1570.869176,home_improvement,63000.0,$60K-$80K
3,68476807,65,Very High,0,0.286666,0.45,10400.0,1341.598081,major_purchase,104433.0,$100K+
4,68426831,50,High,0,0.226972,0.45,11950.0,1220.541094,debt_consolidation,34000.0,< $40K


In [39]:
check_df = pd.read_csv("../data/credit_risk_ecl.csv")

print("Shape:", check_df.shape)
print("\nMissing values:")
print(check_df.isna().sum())

check_df.head()

Shape: (1345350, 11)

Missing values:
loan_id           0
risk_score        0
risk_tier         0
default_flag      0
PD                0
LGD               0
EAD               0
ECL               0
purpose           0
annual_inc        0
income_segment    0
dtype: int64


,loan_id,risk_score,risk_tier,default_flag,PD,LGD,EAD,ECL,purpose,annual_inc,income_segment
0,68407277,30,Medium,0,0.174541,0.45,3600.0,282.756452,debt_consolidation,55000.0,$40K-$60K
1,68355089,15,Low,0,0.111075,0.45,24700.0,1234.599836,small_business,65000.0,$60K-$80K
2,68341763,40,Medium,0,0.174541,0.45,20000.0,1570.869176,home_improvement,63000.0,$60K-$80K
3,68476807,65,Very High,0,0.286666,0.45,10400.0,1341.598081,major_purchase,104433.0,$100K+
4,68426831,50,High,0,0.226972,0.45,11950.0,1220.541094,debt_consolidation,34000.0,< $40K


## Final Dataset Validation

The final credit risk dataset contains 1,345,350 loans with no missing values in the exported risk metrics and segmentation fields.

The dataset combines risk score, risk tier, PD, LGD, EAD, and ECL with loan purpose and borrower income segments. It is ready for portfolio-level analysis and Power BI visualization.